In [1]:
import warnings
warnings.filterwarnings("ignore", message="Attempting to use hipBLASLt")

In [3]:
#%pip install sentence-transformers -q
#%pip install openai -q  # OpenRouter совместим с OpenAI SDK
#%pip install tavily-python
#%pip install langgraph

import os
import numpy as np
import pandas as pd
import time
import json

from openai import OpenAI
from tavily import TavilyClient
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from collections import Counter

### LLM агент с поиском в Интернет. Ансамбль из 3-х LLM моделей. Используем Few-Shot.

Агент строится на фреймворке LangGraph. 

Для поиска в Интернет используем клиента Tavily, который дает 1000 бесплатных запросов в месяц. Чтобы растянуть эту 1000 на всю реализацию проекта, использовал кеширование результатов поиска. Похожесть текстов запроса - по векторной базе, скалярное произведение единичных векторов (минимум 95% => берем из кэша).

Для доступа к LLM используем сервис OpenRouter. Использовал самые недорогие платные модели. Бесплатные - не тянут, либо перегружены и timeout частый. Более дорогие модели наверняка показали бы лучший результат, но у меня нет месячной подписки на фронтирные, а за API платить больше не хотелось. По этой же причине я не использовал рассуждающие (reasoning) модели (что, теоретически, дало бы лучший результат), чтобы уменьшить расход токенов (и денег).


#### АРХИТЕКТУРА АГЕНТА:

Основные узлы - это "опрос LLM" (я использовал не одну LLM, а ансамбль - результат по большинству) и "поиск в Интернет". 

Поиск в Интернет вызывается, если модель его запросила и не был достигнут лимит количества поисков. Если после первого обращения к ансамблю моделей все модели ответили одинаково (есть консенсус), то поиск не выполняется, даже если какие-то из моделей его и запросили - модель сразу дает ответ.

Таким образим возможные циклы работы агента: 

Опрос LLM -> Финиш

Опрос LLM [-> Поиск в Интернет -> Опрос LLM][...] -> Финиш


#### ЭКСПЕРИМЕНТЫ С ПОСТРОЕННЫМ АГЕНТОМ:

1. Первоначально собрал в ансамбль около 10 LLM и прогнал через агента, чтобы понять их пригодность для задачи в принципе. От рассуждающих сразу пришлось отказаться, т.к. им требуется сильно больше 64 токенов - установленное мной ограничение на ответ LLM. Фактически только 3 модели справлялись стабильно - их я и выбрал:
    - "meta-llama/llama-3.3-70b-instruct",
    - "google/gemini-2.5-flash-lite",
    - "deepseek/deepseek-chat-v3-0324"

2. Каждая из 3-х моделей в ансамбле могла запросить поиск в Интернет. На следующей итерации модель, помимо исходного промпта, получала и результаты поиска. И снова модель могла запросить дополнительный поиск в Интернет. В общем случае, так продолжать можно долго, если модели будут продолжать запрашивать поиск. Но я обратил внимание, что эти три модели крайне редко запрашивают повторный поиск, хоть сколько-то отличным запросом от первого запроса на поиск. Поэтому, после нескольких запусков агента, я снизил количество итераций поиска до одной. Т.е. модель либо сразу отвечает, либо один раз получает дополнительную информацию из Интернет, а потом даёт окончательный ответ, всё, нет второго поиска.

3. Варианты с Few-Shot. Примеры выбирал по близости текста запроса, по векторной базе (заполненной по всем текстам запроса "обучающей" выборки). Сначала я давал по два примера с ответом 0, и ответом 2 - всего 4 примера, самых близких. Затем по 1 примеру 0 и 1 - всего два самых близких, т.е. Two-Shot. Стало даже лучше - для этих моделей очень объемный и перегруженный промпт - тяжело не потерять внимание, я так понимаю. Вот в этом блокноте результат запуска именно этой конфигурации (3 LLM в ансамбле и Two-Shot). Не самый успешный запуск - были запуски, которые дали accuracy на 1% выше, чем в этот раз. Но картину показывает. Эффект от применения поиска виден (он не большой, но он есть).

4. Варианты с Zero-Shot. Одна из 3-х использованных моделей, а именно "deepseek/deepseek-chat-v3-0324", с Zero-Shot стала работать очень нестабильно, и примерно в 30% случаев не возвращала ответа по формату. Поэтому для этого эксперимента я оставил только две модели в ансамбле. Логика голосования стала такой: чтобы ансамбль вернул 1, нужно, чтобы обе модели вернули 1, иначе агент возвращает 0. Эти две модели "meta-llama/llama-3.3-70b-instruct" и "google/gemini-2.5-flash-lite" оказались способными с Zero-Shot показать лучший результат, чем ансамбль из 3-х моделей с Two-Shot из третьего эксперимента.

Четвёртый эксперимент можно посмотреть в блокноте: part_3_llm_agent_with_search__m2_with_zero_shot.ipynb.

In [ ]:
# ============================================================
# ЧАСТЬ 1: Конфигурация
# ============================================================

# --- Воспроизводимость ---
SEED = 42
np.random.seed(SEED)

# --- API ключи ---
OPENROUTER_API_KEY = "sk-or-v1-..."
TAVILY_API_KEY     = "tvly-dev-..."

# --- Данные ---
DATA_CSV = "./data/df_data_drop_01.csv"
TEST_CSV = "./data/df_test_drop_01.csv"

# --- Маппинги ---
#NUM_LABELS  = 3
#REL2SCORE   = {0.0: 0, 0.1: 1, 1.0: 2}
#SCORE2REL   = {0: 0.0, 1: 0.1, 2: 1.0}
#SCORE2LABEL = {0: "IRRELEVANT", 1: "RELEVANT_MINUS", 2: "RELEVANT_PLUS"}
NUM_LABELS  = 2
REL2SCORE   = {0.0: 0, 1.0: 1}
SCORE2REL   = {0: 0.0, 1: 1.0} 
SCORE2LABEL = {0: "IRRELEVANT", 1: "RELEVANT"}

# --- Модели ---
ENSEMBLE_MODELS = [
    "meta-llama/llama-3.3-70b-instruct",
    "google/gemini-2.5-flash-lite",
    "deepseek/deepseek-chat-v3-0324",
    #"deepseek/deepseek-v3.2",
    #"deepseek/deepseek-v4-pro",
    #"qwen/qwen-2.5-72b-instruct",
    #"qwen/qwen3.5-9b",
    #"mistralai/mistral-small-3.1-24b-instruct",
    #"z-ai/glm-4.7-flash",
    #"minimax/minimax-m2.5",
    #"tencent/hy3-preview",
]

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# --- OpenRouter ---
RPM_LIMIT = 60
SLEEP_SEC = 60 / RPM_LIMIT

# --- Tavily ---
MAX_SEARCH_ITERATIONS     = 1 # максимум итераций поиска
SEARCH_RESULTS_CACHE_JSON = "./data/search_results_cache.json"

# --- Проверки ---
RUN_TESTS = False

In [5]:
# ============================================================
# ЧАСТЬ 2: Загрузка кэша поисков в Интернете
# ============================================================

# --- Загрузка и сохранение кэше результатов поиска ---
def save_search_cache(search_cache: list[dict]):
    with open(SEARCH_RESULTS_CACHE_JSON, 'w', encoding='utf-8') as f:
        # ensure_ascii=False сохраняет русский текст читаемым, indent=2 для красоты в файле
        json.dump(search_cache, f, ensure_ascii=False, indent=2)

def load_search_cache() -> list[dict]:
    if os.path.exists(SEARCH_RESULTS_CACHE_JSON):
        with open(SEARCH_RESULTS_CACHE_JSON, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

# --- Загружаем кэш результатов поиска ---
search_cache = load_search_cache()

print(f"Элементов в кэше результатов поиска: {len(search_cache)}")

Элементов в кэше результатов поиска: 719


In [6]:
# ============================================================
# ЧАСТЬ 3: Создание клиентов OpenRouter и Tavily
# ============================================================

# --- Клиенты ---
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

tavily_client = TavilyClient(
    api_key=TAVILY_API_KEY,
)

In [7]:
# --- Проверка подключения к OpenRouter ---
if RUN_TESTS:

    print("\nПроверка OpenRouter...")
    test = openrouter_client.chat.completions.create(
        model=ENSEMBLE_MODELS[0],
        messages=[{"role": "user", "content": "Ответь одним словом: привет"}],
        max_tokens=10,
    )
    print(f"OpenRouter OK: {test.choices[0].message.content.strip()}")

In [8]:
# --- Проверка подключения к Tavily ---
if RUN_TESTS:
    
    print("\nПроверка Tavily...")
    test = tavily_client.search(
        "тест", 
        max_results=1
    )
    print(f"Tavily OK: {test['results'][0]['title']}")

In [ ]:
# ============================================================
# ЧАСТЬ 4: Определение класса состояния агента
# ============================================================

class AgentState(TypedDict):
    # Входные данные
    query:             str
    object_data:       dict
    few_shot_examples: list[dict]

    # Результаты оценки ансамбля
    model_responses:   list[dict] # {"model_index": int, "search_iteration": int, "score": 0/1|None, "search_query": str|None, "search_reason": int|None}
    last_responses:    dict[dict] # {<model_index>: {...}}
    # Поиск
    search_iteration:  int        # счётчик итераций
    search_results:    list[dict] # {"model_index": int, "search_iteration": int, "search_query": str, "search_result": str}
    
    # Финальный ответ
    iter0_score:            Optional[int]
    final_score:            Optional[int]
    consensus:              bool
    search_changed_score:   bool

In [10]:
# ============================================================
# ЧАСТЬ 5: Чтение данных
# ============================================================

df_data = pd.read_csv(DATA_CSV)
df_test = pd.read_csv(TEST_CSV)

In [11]:
# ============================================================
# ЧАСТЬ 6: Построение эмбеддингов для few-shot поиска
# ============================================================

# --- Загрузка модели ---
embed_model = SentenceTransformer(EMBED_MODEL)

print(f"Модель загружена: {EMBED_MODEL}")

Модель загружена: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [12]:
# --- Эмбеддируем только поисковые запросы из Data ---
# Только Text — ищем похожие запросы, не объекты
data_queries = df_data["Text"].tolist()

print(f"Строим эмбеддинги для {len(data_queries)} запросов...")
data_embeddings = embed_model.encode(
    data_queries,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,  # нормализация → cosine similarity = dot product
)

print(f"Эмбеддинги построены: {data_embeddings.shape}")  # (N, 384)

Строим эмбеддинги для 29530 запросов...


Batches:   0%|          | 0/116 [00:00<?, ?it/s]

Эмбеддинги построены: (29530, 384)


In [13]:
# --- Проверка ---
if RUN_TESTS:

    # Берём один Test запрос и ищем ближайшие в Data
    sample_query = df_test["Text"].iloc[0]
    sample_embedding = embed_model.encode(
        [sample_query],
        normalize_embeddings=True,
    )

    # Cosine similarity = dot product (после нормализации)
    similarities = data_embeddings @ sample_embedding.T  # (N, 1)
    similarities = similarities.squeeze()

    top5_idx = np.argsort(similarities)[::-1][:5]

    print(f"\nЗапрос: «{sample_query}»")
    print("\nТоп-5 похожих из train:")
    for i, idx in enumerate(top5_idx):
        print(f"  {i+1}. [{similarities[idx]:.3f}] «{df_data['Text'].iloc[idx]}» "
            f"| relevance={df_data['relevance'].iloc[idx]}")

In [14]:
# --- Динамический поиск few-shot примеров ---
def get_few_shot_examples(
    query: str,
    df_data: pd.DataFrame,
    data_embeddings: np.ndarray,
    embed_model: SentenceTransformer,
    top_k: int = 50,
    fallback_k: int = 200,
) -> pd.DataFrame:
    """
    Для заданного запроса находит по два лучших примера
    каждого класса (0.0, 1.0) из Data выборки.

    Стратегия:
    1. Эмбеддируем запрос
    2. Находим top_k ближайших соседей (с дедупликацией по Text)
    3. Берём лучший пример для каждого класса (два лучших)
    4. Если какого-то класса нет — расширяем до fallback_k
    """
    # Эмбеддинг запроса
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True,
    )

    # Cosine similarity со всеми train примерами
    similarities = (data_embeddings @ query_embedding.T).squeeze()

    # Сортируем по убыванию
    sorted_idx = np.argsort(similarities)[::-1]

    def search_in_topk(k: int) -> dict:
        """Ищем лучший пример каждого класса в топ-k."""
        seen_queries = set()
        best_per_class = {}  # {relevance: row}
        second_per_class = {}  # {relevance: row}

        query_lower_case = query.strip().lower()

        for idx in sorted_idx[:k]:
            row = df_data.iloc[idx]
            text = row["Text"]
            relevance = float(row["relevance"])

            text_lower_case = text.strip().lower()

            # Пропускаем дубликаты запросов
            if text_lower_case in seen_queries:
                continue
            seen_queries.add(text_lower_case)

            # Пропускаем сам запрос (точное совпадение)
            if text_lower_case == query_lower_case:
                continue

            # Берём лучший (первый встреченный, затем второй) пример класса
            if relevance not in best_per_class:
                best_per_class[relevance] = row
            elif relevance not in second_per_class:
                second_per_class[relevance] = row

            # Нашли по два примера для всех NUM_LABELS классов — готово
            if len(best_per_class) == NUM_LABELS and len(second_per_class) == NUM_LABELS:
                break

        return best_per_class, second_per_class

    # Сначала ищем в top_k
    best_per_class, second_per_class = search_in_topk(top_k)

    # Если не нашли все классы — расширяем
    if len(best_per_class) < NUM_LABELS or len(second_per_class) < NUM_LABELS:
        best_per_class, second_per_class = search_in_topk(fallback_k)

    # Возвращаем как DataFrame, отсортированный по классу
    best_rows = sorted(best_per_class.values(), key=lambda r: float(r["relevance"]))
    second_rows = sorted(second_per_class.values(), key=lambda r: float(r["relevance"]))
    return pd.DataFrame(best_rows), pd.DataFrame(second_rows)

In [15]:
# --- Проверка ---
if RUN_TESTS:

    sample_query = df_test["Text"].iloc[0]
    print(f"Запрос: «{sample_query}»\n")

    best_df, second_df = get_few_shot_examples(
        query=sample_query,
        df_data=df_data,
        data_embeddings=data_embeddings,
        embed_model=embed_model,
    )

    print(f"Найдено примеров: {len(best_df) + len(second_df)}")
    print()
    for df in [best_df, second_df]:
        for _, row in df.iterrows():
            print(f"[relevance={row['relevance']}]")
            print(f"  Запрос       : {row['Text']}")
            print(f"  Рубрика      : {row['normalized_main_rubric_name_ru']}")
            print(f"  Название     : {row['name']}")
            print(f"  Отзывы       : {str(row['reviews_summarized'])[:100]}...")
            print(f"  Дополнительно: {str(row['prices_summarized'])[:100]}...")
            print()

In [ ]:
# ============================================================
# ЧАСТЬ 7: Узел get_few_shot
# ============================================================

def node_get_few_shot(state: AgentState) -> AgentState:
    """
    Находим few-shot примеры для текущего запроса.
    Переиспользуем get_few_shot_examples из предыдущего ноутбука.
    """
    query = state["query"]

    best_df, second_df = get_few_shot_examples(
        query=query,
        df_data=df_data,
        data_embeddings=data_embeddings,
        embed_model=embed_model,
    )

    # Четыре примера
    #few_shot_df = pd.concat([best_df, second_df], ignore_index=True)

    # Два примера
    few_shot_df = best_df

    # Конвертируем DataFrame → list[dict] для хранения в State
    few_shot_examples = few_shot_df.to_dict(orient="records")

    # Без примеров
    #few_shot_examples = []

    return {"few_shot_examples": few_shot_examples}

In [17]:
# --- Проверка ---
if RUN_TESTS:

    idx = 1
    sample_row   = df_test.iloc[idx]
    print(f"Запрос: «{sample_row.get("Text")}»\n")

    object_data_state = AgentState(
        query=sample_row.get("Text"),

        object_data={
            "name":                           sample_row.get("name"),
            "normalized_main_rubric_name_ru": sample_row.get("normalized_main_rubric_name_ru"),
            "address":                        sample_row.get("address"),
            "reviews_summarized":             sample_row.get("reviews_summarized"),
            "prices_summarized":              sample_row.get("prices_summarized"),
        },

        few_shot_examples=[],
        model_responses=[],
        last_responses={},

        search_iteration=0,
        search_results=[],

        iter0_score=None,
        final_score=None,
        consensus=False,
        search_changed_score=False,
    )

    few_shot_examples_state = node_get_few_shot(object_data_state)

    print(f"Найдено few-shot примеров: {len(few_shot_examples_state['few_shot_examples'])}")
    for ex in few_shot_examples_state["few_shot_examples"]:
        print(f"  [{ex['relevance']}] «{ex['Text']}» | {ex['normalized_main_rubric_name_ru']}")

In [ ]:
# ============================================================
# ЧАСТЬ 8: Построение промпта
# ============================================================

# Системный промпт
SYSTEM_PROMPT_AGENT = """Ты — эксперт (assistant) по оценке релевантности данных объекта тексту запроса пользователя.

Пользователь (user) вводит запрос (query) и ищет подходящий объект. 
Твоя задача — оценить, насколько предоставленные данные объекта соответствуют запросу пользователя.

Верни одну из следующих оценок (score):
0. IRRELEVANT, т.е. объект НЕ соответствует запросу пользователя,
1. RELEVANT, т.е. объект соответствует запросу пользователя.

Если для выбора score требуется дополнительная информация, то сформулируй и верни поисковый запрос (search_query) для уточнения данных.
Включай в search_query название объекта и адрес объекта.

Если информации для выбора score уже достаточно, то search_query верни как null.

Если просишь выполнить поисковый запрос, то укажи причину поиска (search_reason):
1. Запрос содержит специфичный атрибут ("с верандой", "джазовый", "круглосуточный" и т.п.), который не упомянут в данных объекта.
2. Запрос содержит географическое уточнение, которое которое не упомянуто в данных объекта.
3. В данных объекта не хватает информации для принятия решения.

Если search_query равен null, то search_reason ОБЯЗАТЕЛЬНО должен быть null.

Отвечай строго в формате JSON:
{"score": <0 или 1>, "search_query": <str или null>, "search_reason": <1 или 2 или 3 или null>}
Никаких пояснений, только JSON."""

В системный промпт добавлено предложение указывать причину поиска цифровым кодом, выбрав код одной из причин из списка.

Эти коды причин никак не используются, но, похоже, что помогают направить "мысль" модели при принятии решения о поисковом запросе и его формулировке.

Первоначально я перечислил в промпте больше причин, но оставил только эти три, которые модели чаще выбирали. Третья причина - это как "причие причины".

Пробовал удалять их из системного промпта - но точность незначительно, но снижалась. Поэтому перечисление причин поиска оставил в промпте.

In [ ]:
def smart_str_cut(long_str, max_length):
    long_str = long_str.replace('\n', ' ')

    if len(long_str) <= max_length:
        return long_str
    
    # Обрезаем до последнего пробела в пределах max_length символов
    cut_str = long_str[:max_length].rsplit(" ", 1)[0] + "..."
    return cut_str


def format_object(row: pd.Series) -> str:
    """
    Форматируем объект для промпта.
    """
    # Берём только первое название (до ";")
    #name    = smart_str_cut(str(row.get("name", "")).split(";")[0].strip(), 200)
    name    = smart_str_cut(str(row.get("name", "")).strip(), 200)
    rubric  = smart_str_cut(str(row.get("normalized_main_rubric_name_ru", "")).strip(), 200)
    address = smart_str_cut(str(row.get("address", "")).strip(), 200)

    reviews = smart_str_cut(str(row.get("reviews_summarized", "")).strip(), 300)
    inform  = smart_str_cut(str(row.get("prices_summarized", "")).strip(), 300)

    if len(inform) < 50:
        inform = reviews

    return (
        f"Рубрика: {rubric}\n"
        f"Название: {name}\n"
        f"Адрес: {address}\n"
        f"Информация: {inform}"
    )

In [20]:
def build_agent_prompt(
    query: str,
    object_data: dict,
    few_shot_examples: list[dict],
    search_history: list[dict],  # результаты поиска для данной модели
) -> list[dict]:
    """
    Строим промпт для одной модели ансамбля.
    Включаем few-shot примеры и историю поиска если есть.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT_AGENT}]

    # Few-shot примеры
    for ex in few_shot_examples:
        # user content
        user_content = f"Запрос: {ex['Text']}\n\n{format_object(pd.Series(ex))}"
        messages.append({"role": "user", "content": user_content})
        # assistant content
        score = REL2SCORE[(ex["relevance"])]  # теперь 0.0/1.0 → 0/1, а не 0.0/0.1/1.0 → 0/1/2
        assistant_content = json.dumps(
            {"score": score, "search_query": None, "search_reason": None},
            ensure_ascii=False,
        )
        messages.append({"role": "assistant", "content": assistant_content})

    # Целевой объект
    user_content = f"Запрос: {query}\n\n{format_object(pd.Series(object_data))}"

    # Добавляем историю поиска если есть
    if search_history:
        search_text = "\n\nДополнительная информация из поиска в интернете:\n"
        for s in search_history:
            search_text += f"[Итерация {s['search_iteration']}] Запрос: «{s['search_query']}»\nРезультат: {s['search_result']}\n"
        #search_text += "\nЕсли дополнительной информации из поиска не хватает для принятия уверенного решения, то возвращай оценку 0 (IRRELEVANT)\n"
        user_content += search_text

    messages.append({"role": "user", "content": user_content})
    return messages

In [ ]:
# ============================================================
# ЧАСТЬ 9: Узел ensemble_evaluate
# ============================================================

def clean_json(raw: str) -> str:
    """
    Убираем markdown-обёртку если есть.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        # Убираем первую строку (```json или ```) и последнюю (```)
        lines = raw.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        raw = "\n".join(lines).strip()
    return raw

def node_ensemble_evaluate(state: AgentState) -> AgentState:
    """
    Последовательно опрашиваем все модели ансамбля.
    Каждая модель получает свою историю поиска.
    """
    query             = state["query"]
    object_data       = state["object_data"]
    few_shot_examples = state["few_shot_examples"]
    model_responses   = list(state["model_responses"])
    last_responses    = dict(state["last_responses"])
    search_iteration  = state["search_iteration"]
    search_results    = state["search_results"]

    print(f"[llms query] Итерация {search_iteration}, моделей в ансамбле {len(ENSEMBLE_MODELS)}")

    for model_index, model_id in enumerate(ENSEMBLE_MODELS):
        # История поиска для данной модели
        search_history = [
            {
                "search_iteration": r["search_iteration"],
                "search_query":     r["search_query"],
                "search_result":    r["search_result"],
            }
            for r in search_results
            if r["model_index"] == model_index
        ]

        # Не выполняем повторный опрос модели, если нет новых данных поиска
        if search_iteration > 0:
            curr_iter_search_history = [
                {
                    "search_iteration": r["search_iteration"],
                    "search_query":     r["search_query"],
                    "search_result":    r["search_result"],
                }
                for r in search_history
                if r["search_iteration"] == search_iteration
            ]        
            if len(curr_iter_search_history) == 0:
                continue            

        # Строим запрос к модели
        messages = build_agent_prompt(
            query=query,
            object_data=object_data,
            few_shot_examples=few_shot_examples,
            search_history=search_history,
        )

        # Выполняем запрос к модели с retry
        response = None
        for attempt in range(3):
            try:
                response_obj = openrouter_client.chat.completions.create(
                    model=model_id,
                    messages=messages,
                    temperature=0.0,
                    max_tokens=64,
                )

                raw = str(response_obj.choices[0].message.content).strip()
                #print(raw)   
                             
                parsed = json.loads(clean_json(raw))
                score  = int(parsed["score"])

                if score not in (0, 1):
                    raise ValueError(f"Неожиданный score: {score}")

                response = {
                    "model_index":      model_index,
                    "score":            score,
                    "search_iteration": search_iteration,
                    "search_query":     parsed.get("search_query"),
                    "search_reason":    parsed.get("search_reason"),
                }
                break

            except Exception as e:
                print(f"  [model {model_index}, attempt {attempt+1}] Ошибка: {e}")
                time.sleep(2 ** attempt)

        if response is None:
            # Если все попытки неудачны — score None
            response = {
                "model_index":      model_index,
                "score":            None,
                "search_iteration": search_iteration,
                "search_query":     None,
                "search_reason":    None,
            }
            # Если это нулевая итерация, то включаем и неудачную попытку
            if model_index not in last_responses:
                last_responses[model_index] = response    
        else:
            # Удачную попытку фиксируем в last_responses всегда безусловно
            last_responses[model_index] = response

        model_responses.append(response)

        time.sleep(SLEEP_SEC)

    return {"model_responses": model_responses, "last_responses": last_responses}

In [22]:
# --- Проверка ---
if RUN_TESTS:

    print("--- Тест node_ensemble_evaluate ---")
    model_responses_state = node_ensemble_evaluate({
        **object_data_state, 
        **few_shot_examples_state,
    })

    print(f"\nВсего ответов: {len(model_responses_state['model_responses'])}")
    for r in model_responses_state["model_responses"]:
        print(f"  {r}")

In [23]:
# ============================================================
# ЧАСТЬ 10: Узел check_consensus и функция маршрутизации
# ============================================================

def get_current_scores(state: AgentState) -> list[int | None]:
    """
    Возвращает scores всех моделей на последней (успешной) итерации для каждой модели.
    Включаем None (если первое же обращение оказалось неудачным) — они важны для проверки консенсуса.
    """
    
    #latest_responses = {}
    #for r in state["model_responses"]:
    #    latest_responses[r["model_index"]] = r
    #scores = [r["score"] for r in latest_responses.values()]

    return [r["score"] for r in state["last_responses"].values()]



def node_check_consensus(state: AgentState) -> AgentState:
    """
    Консенсус = все модели ответили (нет None) И все совпали.
    """
    scores = get_current_scores(state)

    # Все ли модели ответили
    all_answered = (
        len(scores) == len(ENSEMBLE_MODELS) and
        all(s is not None for s in scores)
    )

    if all_answered and len(set(scores)) == 1: # and (len(ENSEMBLE_MODELS) > 1 or state["last_responses"][0]["search_query"] is None):
        return {"consensus": True, "final_score": scores[0]}

    return {"consensus": False}


def route_after_check_consensus(state: AgentState) -> str:
    """Условное ребро после check_consensus."""
    if state["consensus"]:
        return "finish"
    return "check_search_iteration"

In [24]:
# --- Проверка ---
if RUN_TESTS:

    # Тест 1: консенсус есть
    state_consensus = {
        **object_data_state,
        "search_iteration": 0,
        "last_responses": {
            0: {"model_index": 0, "search_iteration": 0, "score": 1, "search_query": None, "search_reason": None},
            1: {"model_index": 1, "search_iteration": 0, "score": 1, "search_query": None, "search_reason": None},
            2: {"model_index": 2, "search_iteration": 0, "score": 1, "search_query": None, "search_reason": None},
        },
    }
    result = node_check_consensus(state_consensus)
    print(f"Тест 1 (консенсус): consensus={result['consensus']}, final_score={result.get('final_score')}")
    print(f"  Маршрут: {route_after_check_consensus({**state_consensus, **result})}")

    # Тест 2: нет консенсуса — разные мнения
    state_no_consensus = {
        **object_data_state,
        "search_iteration": 0,
        "last_responses": {
            0: {"model_index": 0, "search_iteration": 0, "score": 0, "search_query": "запрос", "search_reason": 1},
            1: {"model_index": 1, "search_iteration": 0, "score": 1, "search_query": None,    "search_reason": None},
            2: {"model_index": 2, "search_iteration": 0, "score": 0, "search_query": "запрос", "search_reason": 1},
        },
    }
    result = node_check_consensus(state_no_consensus)
    print(f"\nТест 2 (разные мнения): consensus={result['consensus']}")
    print(f"  Маршрут: {route_after_check_consensus({**state_no_consensus, **result})}")

    # Тест 3: нет консенсуса — одна модель вернула None
    state_none = {
        **object_data_state,
        "search_iteration": 0,
        "last_responses": {
            0: {"model_index": 0, "search_iteration": 0, "score": 1,    "search_query": None, "search_reason": None},
            1: {"model_index": 1, "search_iteration": 0, "score": 1,    "search_query": None, "search_reason": None},
            2: {"model_index": 2, "search_iteration": 0, "score": None, "search_query": None, "search_reason": None},
        },
    }
    result = node_check_consensus(state_none)
    print(f"\nТест 3 (одна модель None): consensus={result['consensus']}")
    print(f"  Маршрут: {route_after_check_consensus({**state_none, **result})}")

In [25]:
# ============================================================
# ЧАСТЬ 11: Узел check_search_iteration и функция маршрутизации
# ============================================================

def get_majority_score(state: AgentState, search_iteration: int = -1) -> int:
    """
    Берём score большинства на последней итерации.
    При ничьей (например 1:1 при двух моделях) берём 0 (IRRELEVANT) —
    лучше перестраховаться чем показать нерелевантный результат.
    """

    if search_iteration == -1:
        # Последние ответы каждой из моделей
        scores = [
            r["score"]
            for r in state["last_responses"].values()
            if r["score"] is not None
        ]
    else:
        # Переданная в параметре итерация
        scores = [
            r["score"]
            for r in state["model_responses"]
            if r["search_iteration"] == search_iteration and r["score"] is not None
        ]

    if not scores:
        return 0

    # Считаем голоса
    counter = Counter(scores)
    # Сортируем по убыванию количества голосов, при ничьей берём 0
    majority = counter.most_common()
    if len(majority) > 1 and majority[0][1] == majority[1][1]:
        return 0  # ничья → IRRELEVANT

    return majority[0][0]


def node_check_search_iteration(state: AgentState) -> AgentState:
    """
    Проверяем не исчерпан ли лимит итераций поиска.
    Если лимит достигнут — записываем final_score через большинство.
    """
    if state["search_iteration"] >= MAX_SEARCH_ITERATIONS:
        majority = get_majority_score(state)
        return {"final_score": majority}
    
    # Собираем запросы моделей которые попросили поиск на текущей итерации
    current_iteration = state["search_iteration"]
    queries_to_search = [
        {
            "model_index":  r["model_index"],
            "search_query": r["search_query"],
        }
        for r in state["model_responses"]
        if r["search_iteration"] == current_iteration
        and r["search_query"] is not None
    ]

    # Если ни одна модель не запросила поиск, то считаем final_score
    if len(queries_to_search) == 0:
        majority = get_majority_score(state)
        return {"final_score": majority}  
    
    return {}


def route_after_check_search_iteration(state: AgentState) -> str:
    """Условное ребро после check_search_iteration."""
    if state["final_score"] is not None:
        return "finish"
    return "web_search"

In [26]:
# --- Проверка ---
if RUN_TESTS:

    # Тест 1: лимит не достигнут → идём в web_search
    state_iter_0 = {
        **model_responses_state, 
        "search_iteration": 0, 
        "final_score": None
    }
    result = node_check_search_iteration(state_iter_0)
    print(f"Тест 1 (search_iteration=0): result={result}")
    print(f"  Маршрут: {route_after_check_search_iteration({**state_iter_0, **result})}")

    # Тест 2: лимит достигнут → финиш с большинством
    state_iter_2 = {
        **object_data_state,
        "search_iteration": 2,
        "final_score": None,
        "last_responses": {
            0: {"model_index": 0, "search_iteration": 2, "score": 1, "search_query": None, "search_reason": None},
            1: {"model_index": 1, "search_iteration": 2, "score": 0, "search_query": None, "search_reason": None},
            2: {"model_index": 2, "search_iteration": 2, "score": 1, "search_query": None, "search_reason": None},
        },
    }
    result = node_check_search_iteration(state_iter_2)
    print(f"\nТест 2 (search_iteration=2, счёт 2:1): result={result}")
    print(f"  Маршрут: {route_after_check_search_iteration({**state_iter_2, **result})}")

    # Тест 3: лимит достигнут → ничья → IRRELEVANT
    state_iter_2_tie = {
        **object_data_state,
        "search_iteration": 2,
        "final_score": None,
        "last_responses": {
            0: {"model_index": 0, "search_iteration": 2, "score": 1, "search_query": None, "search_reason": None},
            1: {"model_index": 1, "search_iteration": 2, "score": 0, "search_query": None, "search_reason": None},
        },
    }
    result = node_check_search_iteration(state_iter_2_tie)
    print(f"\nТест 3 (search_iteration=2, ничья 1:1): result={result}")
    print(f"  Маршрут: {route_after_check_search_iteration({**state_iter_2_tie, **result})}")

In [27]:
# ============================================================
# ЧАСТЬ 12: Узел web_search
# ============================================================

def check_search_cache(search_query: str, search_cache: list[dict]) -> dict:
    SIMILARITY_THRESHOLD_VERY   = 0.98 # Очень похожий запрос
    SIMILARITY_THRESHOLD_ENOUGH = 0.95 # Достаточно похожий запрос

    # Эмбеддируем все запрос
    embedding = embed_model.encode(
        search_query,
        normalize_embeddings=True,
    )    

    # Ищем подобные в кэше
    best_data = None
    best_sim = 0.0 
    for cached_data in search_cache:
        sim = float(embedding @ np.array(cached_data["embedding"])) 
        # Если нашли очень похожий, то сразу возвращаем его   
        if sim >= SIMILARITY_THRESHOLD_VERY:
            return {
                "found_in_cache": True,
                "similarity": sim,
                "search_query": cached_data["search_query"],
                "search_result": cached_data["search_result"],
            }
        # Если нашли более похожий, чем до этого, то запоминаем его
        if sim >= SIMILARITY_THRESHOLD_ENOUGH and sim > best_sim:
            best_data = {
                "found_in_cache": True,
                "similarity": sim,
                "search_query": cached_data["search_query"],
                "search_result": cached_data["search_result"],
            }

    # Если был найден достаточно похожий, то возвращаем его
    if best_data is not None:
        return best_data
    
    # Если в кэше нет похожих, то возвращаем embedding для последующего добавления в кэш
    return {
        "found_in_cache": False,
        "search_query": search_query,
        "embedding": embedding.tolist(),
    }

In [ ]:
def node_web_search(state: AgentState) -> AgentState:
    current_iteration = state["search_iteration"]
    next_iteration    = current_iteration + 1

    # Собираем запросы моделей которые попросили поиск на текущей итерации
    queries_to_search = [
        {
            "model_index":  r["model_index"],
            "search_query": r["search_query"],
        }
        for r in state["model_responses"]
        if r["search_iteration"] == current_iteration
        and r["search_query"] is not None
    ]

    print(f"[web search] Итерация {next_iteration}, запросов от моделей {len(queries_to_search)}")

    if not queries_to_search:
        # Никто не попросил поиск — просто инкрементируем итерацию
        return {"search_iteration": next_iteration}

    search_cache_updated = False   
    new_search_results   = list(state["search_results"]) 
    
    for q in queries_to_search:
        search_query = q["search_query"]
        cached_data = check_search_cache(search_query, search_cache)

        if cached_data["found_in_cache"]:
            # Если поиск в кэше дал результат
            print(f"  [model {q['model_index']}, from cache]: «{search_query}» -> ({cached_data["similarity"]:.2f}) -> «{cached_data["search_query"]}»")
            new_search_results.append({
                "model_index":      q["model_index"],
                "search_iteration": next_iteration,
                "search_query":     cached_data["search_query"],
                "search_result":    cached_data["search_result"],
            })

        else:
            # Если в кэше ничего похожего не нашли, то ищем в Интернет
            print(f"  [model {q['model_index']}]: «{search_query}»")
            try:
                tavily_response = tavily_client.search(
                    q["search_query"],
                    max_results=3,
                    include_raw_content=False,
                )
                # Объединяем сниппеты в одну строку
                search_result = " | ".join(
                    r["content"] for r in tavily_response["results"] if r.get("content")
                )
            except Exception as e:
                print(f"Ошибка Tavily: {e}")
                search_result = ""

            new_search_results.append({
                "model_index":      q["model_index"],
                "search_iteration": next_iteration,
                "search_query":     search_query,
                "search_result":    search_result,
            })

            # Если поиск был результативным, то добавляем его результаты в кэш
            if len(search_result) > 0:
                search_cache.append({
                    "embedding":     cached_data["embedding"],
                    "search_query":  search_query,
                    "search_result": search_result,
                }) 
                search_cache_updated = True

    if search_cache_updated:
        save_search_cache(search_cache) 
        print("  [search cache saved]")              

    return {
        "search_iteration": next_iteration,
        "search_results":   new_search_results,
    }

In [29]:
# --- Проверка ---
if RUN_TESTS:

    search_results_state = node_web_search({
        **object_data_state,
        **few_shot_examples_state, 
        **model_responses_state, 
        #"model_responses": [
        #    {"model_index": 0, "search_iteration": 0, "score": 0,
        #     "search_query": "кальянная спб мероприятия", "search_reason": 1},
        #    {"model_index": 1, "search_iteration": 0, "score": 0,
        #     "search_query": "кальянная мероприятия СПб", "search_reason": 1},
        #    {"model_index": 2, "search_iteration": 0, "score": 1,
        #     "search_query": None, "search_reason": None},
        #],
        "search_iteration": 0,
        "search_results": [],
    })

    print(f"\nНовая итерация: {search_results_state['search_iteration']}")
    print(f"search_results: {len(search_results_state['search_results'])}")
    for r in search_results_state["search_results"]:
        print(f"\n  model_index={r['model_index']}, search_iteration={r['search_iteration']}")
        print(f"  result: {r['search_result'][:150]}...")

In [30]:
# ============================================================
# ЧАСТЬ 13: Узел finish
# ============================================================

def node_finish(state: AgentState) -> AgentState:
    """
    Финальный узел — определяем, был ли эффект от поиска
    """

    iter0_score = get_majority_score(state, search_iteration=0)
    search_changed_score = (iter0_score != state['final_score'])

    return {
        "iter0_score":          iter0_score, 
        "search_changed_score": search_changed_score,
    }

In [31]:
# --- Проверка ---
if RUN_TESTS:

    state_finish_1 = node_finish({
        **model_responses_state,
        "final_score": 1,
        "consensus":   True,
        "search_iteration":   0,
    })
    print(state_finish_1)

    state_finish_2 = node_finish({
        **model_responses_state,
        "final_score": 0,
        "consensus":   False,
        "search_iteration":   2,
    })
    print(state_finish_2)

In [32]:
# ============================================================
# ЧАСТЬ 14: Сборка графа
# ============================================================

builder = StateGraph(AgentState)

# --- Узлы ---
builder.add_node("get_few_shot",           node_get_few_shot)
builder.add_node("ensemble_evaluate",      node_ensemble_evaluate)
builder.add_node("check_consensus",        node_check_consensus)
builder.add_node("check_search_iteration", node_check_search_iteration)
builder.add_node("web_search",             node_web_search)
builder.add_node("finish",                 node_finish)

# --- Рёбра ---
builder.add_edge(START,                    "get_few_shot")
builder.add_edge("get_few_shot",           "ensemble_evaluate")
builder.add_edge("ensemble_evaluate",      "check_consensus")

builder.add_conditional_edges(
    "check_consensus",
    route_after_check_consensus,
    {
        "finish":                          "finish",
        "check_search_iteration":          "check_search_iteration",
    }
)

builder.add_conditional_edges(
    "check_search_iteration",
    route_after_check_search_iteration,
    {
        "finish":                          "finish",
        "web_search":                      "web_search",
    }
)

builder.add_edge("web_search",             "ensemble_evaluate")
builder.add_edge("finish",                 END)

# --- Компиляция ---
graph = builder.compile()
print("Граф скомпилирован успешно")

Граф скомпилирован успешно


In [33]:
# ============================================================
# ЧАСТЬ 15: Тест на одном примере
# ============================================================

# Берём пример из Test выборки
idx = 0
sample_row = df_test.iloc[idx]

initial_state = AgentState(
    query=sample_row["Text"],

    object_data={
        "name":                           sample_row["name"],
        "normalized_main_rubric_name_ru": sample_row["normalized_main_rubric_name_ru"],
        "address":                        sample_row["address"],
        "reviews_summarized":             sample_row["reviews_summarized"],
        "prices_summarized":              sample_row["prices_summarized"],
    },

    few_shot_examples=[],

    model_responses=[],
    last_responses={},

    search_iteration=0,
    search_results=[],

    iter0_score=None,
    final_score=None,
    consensus=False,
    search_changed_score=False,
)

print("\n--- Запуск агента ---")
result = graph.invoke(initial_state)


--- Запуск агента ---
[llms query] Итерация 0, моделей в ансамбле 3
[web search] Итерация 1, запросов от моделей 2
  [model 0]: «сигары Tabaccos Москва Дубравная улица 34/29»
  [model 2, from cache]: «Tabaccos, Москва, Дубравная улица, 34/29» -> (1.00) -> «Tabaccos, Москва, Дубравная улица, 34/29»
  [search cache saved]
[llms query] Итерация 1, моделей в ансамбле 3


In [34]:
print("\n--- Данные объекта ---")
print(f"Название:       {sample_row['name']}")
print(f"Рубрика:        {sample_row['normalized_main_rubric_name_ru']}")
print(f"Адрес:          {sample_row['address']}")
#print(f"Отзывы:         {sample_row['reviews_summarized']}")
#print(f"Дополнительно:  {sample_row['prices_summarized']}")


print(f"\nЗапрос:       {sample_row['Text']}")

print(f"\n--- Итог ---")
search_used = len(result["search_results"]) > 0
search_changed_score = result["search_changed_score"]

matched = result["final_score"] == REL2SCORE[sample_row["relevance"]]
search_helped = search_changed_score and matched
search_harmed = search_changed_score and not matched 

print(f"final_score:            {result['final_score']} ({SCORE2LABEL[result['final_score']]})")
print(f"consensus:              {result['consensus']}")
print(f"search_used:            {search_used}")
print(f"search_changed_score:   {search_changed_score}\n")

print(f"true_score:             {REL2SCORE[sample_row['relevance']]} ({SCORE2LABEL[REL2SCORE[sample_row['relevance']]]})")
print(f"matched:                {matched}")
print(f"search_helped:          {search_helped}")
print(f"search_harmed:          {search_harmed}")

print(f"\nПоследние ответы каждой из моделей:")
for r in result["last_responses"].values():
    print(f"  {r}")

print(f"\nВсе ответы моделей:")
for r in result["model_responses"]:
    print(f"  {r}")


--- Данные объекта ---
Название:       Tabaccos; Магазин Tabaccos; Табаккос
Рубрика:        Магазин табака и курительных принадлежностей
Адрес:          Москва, Дубравная улица, 34/29

Запрос:       сигары

--- Итог ---
final_score:            1 (RELEVANT)
consensus:              True
search_used:            True
search_changed_score:   True

true_score:             1 (RELEVANT)
matched:                True
search_helped:          True
search_harmed:          False

Последние ответы каждой из моделей:
  {'model_index': 0, 'score': 1, 'search_iteration': 1, 'search_query': None, 'search_reason': None}
  {'model_index': 1, 'score': 1, 'search_iteration': 0, 'search_query': None, 'search_reason': None}
  {'model_index': 2, 'score': 1, 'search_iteration': 1, 'search_query': None, 'search_reason': None}

Все ответы моделей:
  {'model_index': 0, 'score': 0, 'search_iteration': 0, 'search_query': 'сигары Tabaccos Москва Дубравная улица 34/29', 'search_reason': 1}
  {'model_index': 1, 'score'

In [35]:
result["search_results"]

[{'model_index': 0,
  'search_iteration': 1,
  'search_query': 'сигары Tabaccos Москва Дубравная улица 34/29',
  'search_result': 'Oфициальный сайт. tabaccos.ru. Адрес. Москва, Митино, Дубравная улица, 34/29 Посмотреть на карте. Отзывы. 0. Хороших. : 0. Плохих. Оставить отзыв. Отзывы. 0. | Магазин табака и курительных принадлежностей Tabaccos по адресу Москва, Дубравная улица, 34/29, метро «Митино», ☎️ показать телефоны. Читать 58 отзывов | Адрес Москва, ул. Дубравная, 34/29, ТРЦ Ладья, 1 этаж ; Телефон +7 (999) 849-58-30 ; Время работы ежедневно 09:30-23:00 ; Официальный сайт. http://tabaccos.ru.'},
 {'model_index': 2,
  'search_iteration': 1,
  'search_query': 'Tabaccos, Москва, Дубравная улица, 34/29',
  'search_result': '# Tabaccos. Точное время работы и предоставляемые услуги можно узнать по номеру +7 (939) 555-48-09 или на сайте tabaccos.pro. Дубравная, 34/29, ТРЦ Ладья, 1 этаж, Митино район. Митино (2 мин), Пятницкое шоссе (13 мин), Волоколамская (23 мин). ежедневно 09:00-23:00.

In [36]:
sample_row

Text                                                                         сигары
address                                              Москва, Дубравная улица, 34/29
name                                           Tabaccos; Магазин Tabaccos; Табаккос
normalized_main_rubric_name_ru         Магазин табака и курительных принадлежностей
permalink                                                                1263329400
prices_summarized                                                               NaN
reviews_summarized                Организация занимается продажей табака, курите...
relevance                                                                       1.0
Name: 0, dtype: object

In [37]:
# ============================================================
# ЧАСТЬ 16: Подготовка выборки для теста
# ============================================================

# Test выборка большая (500 строк)
# Можно сделать стратифицированную подвыборку (для экономии денег и времени)
SAMPLE_SIZE = None # None = вся Test выборка

if SAMPLE_SIZE is not None:
    _, df_eval = train_test_split(
        df_test,
        test_size=SAMPLE_SIZE,
        stratify=df_test["relevance"],
        random_state=SEED,
    )
    df_eval = df_eval.reset_index(drop=True)
    print(f"Используем подвыборку: {len(df_eval)} строк")
    print(df_eval["relevance"].value_counts(normalize=True).round(3))
else:
    df_eval = df_test.reset_index(drop=True)
    print(f"Используем всю Test выборку: {len(df_eval)} строк")

Используем всю Test выборку: 500 строк


In [38]:
# ============================================================
# ЧАСТЬ 17: Тестирование агента
# ============================================================

def ask_agent_with_search(df):
    results = []
    right_answers = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="LLM agents with search inference"):
        
        initial_state = AgentState(
            query=row["Text"],

            object_data={
                "name":                           row["name"],
                "normalized_main_rubric_name_ru": row["normalized_main_rubric_name_ru"],
                "address":                        row["address"],
                "reviews_summarized":             row["reviews_summarized"],
                "prices_summarized":              row["prices_summarized"],
            },

            few_shot_examples=[],

            model_responses=[],
            last_responses={},

            search_iteration=0,
            search_results=[],

            iter0_score=None,
            final_score=None,
            consensus=False,
            search_changed_score=False,
        )

        print(f"\n--- Запуск агента (#{idx + 1})")
        print(f"--- Запрос пользователя:  «{row["Text"]}»")
        print(f"--- Наименование объекта: «{row["name"]}»")
        print(f"--- Адрес объекта:        «{row["address"]}»")

        result = graph.invoke(initial_state)

        search_used = len(result["search_results"]) > 0
        search_changed_score = result["search_changed_score"]

        final_score = result["final_score"]
        true_score = REL2SCORE[row["relevance"]]
        matched = final_score == true_score

        consensus = result['consensus']

        search_helped = search_changed_score and matched
        search_harmed = search_changed_score and not matched 

        object_data = result["object_data"]

        results.append({
            "idx":                  idx,
            "query":                result["query"],
            "name":                 object_data["name"],
            "address":              object_data["address"],

            "iter0_score":          result["iter0_score"],
            "final_score":          final_score,
            "consensus":            consensus,
            "search_used":          search_used,
            "search_changed_score": search_changed_score,

            "true_score":           true_score,
            "matched":              matched,
            "search_helped":        search_helped,
            "search_harmed":        search_harmed,

            "result":               result,
        })

        if matched:
            right_answers += 1

        print(f"--- Есть совпадение: {matched}, есть консенсус: {consensus}, ответ агента: {final_score},"
            f" правильный ответ: {true_score}, достигнутая точность: {right_answers/(idx + 1):.4f}")
        
        if not consensus:
            for model_index, model_response in result["last_responses"].items():
                if model_response["score"] != final_score:
                    print(f"--- Несогласная модель в ансамбле: {model_index} ({ENSEMBLE_MODELS[model_index]})")
            
        if search_changed_score:
            if matched:
                print("--- Поиск изменил ответ агента на правильный (помог) ---")  
            else:          
                print("--- Поиск изменил ответ агента на неправильный (помешал) ---")  

        # Rate limiting
        time.sleep(SLEEP_SEC)

    return results   

In [39]:
results = ask_agent_with_search(df_eval)

LLM agents with search inference:   0%|          | 0/500 [00:00<?, ?it/s]


--- Запуск агента (#1)
--- Запрос пользователя:  «сигары»
--- Наименование объекта: «Tabaccos; Магазин Tabaccos; Табаккос»
--- Адрес объекта:        «Москва, Дубравная улица, 34/29»
[llms query] Итерация 0, моделей в ансамбле 3
[web search] Итерация 1, запросов от моделей 1
  [model 2, from cache]: «Tabaccos, Москва, Дубравная улица, 34/29» -> (1.00) -> «Tabaccos, Москва, Дубравная улица, 34/29»
[llms query] Итерация 1, моделей в ансамбле 3
--- Есть совпадение: True, есть консенсус: False, ответ агента: 1, правильный ответ: 1, достигнутая точность: 1.0000
--- Несогласная модель в ансамбле: 2 (deepseek/deepseek-chat-v3-0324)

--- Запуск агента (#2)
--- Запрос пользователя:  «кальянная спб мероприятия»
--- Наименование объекта: «PioNero; Pionero; Пицца Паста бар; Pio Nero; Pio Nerо; Pizza Pasta Bar»
--- Адрес объекта:        «Санкт-Петербург, Большой проспект Петроградской стороны, 20/5»
[llms query] Итерация 0, моделей в ансамбле 3
[web search] Итерация 1, запросов от моделей 2
  [mode

In [ ]:
# ============================================================
# ЧАСТЬ 18: Оценка результатов работы агента
# ============================================================

df_results = pd.DataFrame(results)

# --- Статистика по ошибкам парсинга ---
nan_condition = df_results["iter0_score"].isna() | df_results["final_score"].isna()
none_count = nan_condition.sum()
print(f"Ошибок (None): {none_count} из {len(df_results)}")

# Убираем строки где LLM не смогла дать ответ (None)
df_clean = df_results.dropna(subset=["iter0_score", "final_score"]).copy()

print(f"Всего примеров : {len(df_results)}")
print(f"После очистки  : {len(df_clean)}")

Ошибок (None): 0 из 500
Всего примеров : 500
После очистки  : 500


Все 500 тестовых объектов были оценены агентом.

Посмотрим сначала на точность, полученную на нулевой итерации, до возможного обращения к поиску. 

In [41]:
# --- Метрики до использования результатов поиска ---
true_scores = df_clean["true_score"].values
pred_scores = df_clean["iter0_score"].values

f1_macro = f1_score(true_scores, pred_scores, average="macro")
print(f"\nF1-macro: {f1_macro:.4f}")

print("\n--- Classification Report (without search) ---")
print(classification_report(
    true_scores,
    pred_scores,
    target_names=list(SCORE2LABEL.values()),
    digits=4,
))


F1-macro: 0.7053

--- Classification Report (without search) ---
              precision    recall  f1-score   support

  IRRELEVANT     0.5688    0.8361    0.6770       183
    RELEVANT     0.8701    0.6341    0.7336       317

    accuracy                         0.7080       500
   macro avg     0.7195    0.7351    0.7053       500
weighted avg     0.7598    0.7080    0.7129       500



In [42]:
# --- Количество ошибок и точность каждой отдельной модели ---
counters = {}
for model_index in range(len(ENSEMBLE_MODELS)):
    counters[model_index] = 0

for index, row in df_clean.iterrows():
    true_score = row["true_score"]
    result = row["result"]
    for model_index, model_response in result["last_responses"].items():
        if model_response["score"] != true_score:
            counters[model_index] += 1

for model_index, times_failed in sorted(counters.items()):
    print(f"Модель {model_index} ошиблась {times_failed} раз(а), точность: {(len(df_clean) - times_failed)/len(df_clean):.4f} ({ENSEMBLE_MODELS[model_index]})")

Модель 0 ошиблась 140 раз(а), точность: 0.7200 (meta-llama/llama-3.3-70b-instruct)
Модель 1 ошиблась 135 раз(а), точность: 0.7300 (google/gemini-2.5-flash-lite)
Модель 2 ошиблась 167 раз(а), точность: 0.6660 (deepseek/deepseek-chat-v3-0324)


Точность менее 71%. Это не самый удачный прогон - модели много ошибались, особенно deepseek (менее 67%). 

Лучший результат (точность) llama, из тех что у меня получались за время работы с агентом, достигал около 76%. 

А в этом прогоне llama показала только 72%, но эта индивидуальная точность всё равно превзошла точность ансамбля (менее 71%).

В половине прогонов ансамбль у меня показывал точность чуть лучше, чем лучшая модель в нем. В это раз deepseek всё испортил.

Теперь посмотрим, как повлиял на результат агента поиск в Интернет.

In [43]:
# --- Метрики после использования результатов поиска ---
true_scores = df_clean["true_score"].values
pred_scores = df_clean["final_score"].values

f1_macro = f1_score(true_scores, pred_scores, average="macro")
print(f"\nF1-macro: {f1_macro:.4f}")

print("\n--- Classification Report (with search) ---")
print(classification_report(
    true_scores,
    pred_scores,
    target_names=list(SCORE2LABEL.values()),
    digits=4,
))


F1-macro: 0.7275

--- Classification Report (with search) ---
              precision    recall  f1-score   support

  IRRELEVANT     0.6042    0.7923    0.6856       183
    RELEVANT     0.8538    0.7003    0.7695       317

    accuracy                         0.7340       500
   macro avg     0.7290    0.7463    0.7275       500
weighted avg     0.7625    0.7340    0.7388       500



Точность агента с менее 71% подросла до более 73%.

Поиск использовался примерно каждый пятый запуск агента (4 из 5 обходились без поиска - модели не запрашивали).

Поиск только в 30% случаев использования повлиял на решение ансамбля моделей, в 70% случаев результаты поиска не изменили предварительного решения ансамбля.

Причем не всегда это было полезно. Но всё же чаще результаты поиска шли на пользу, чем наоборот.

In [44]:
# --- Статистика использования поиска ---

# 1. Сколько всего раз был использован поиск
total_searches = df_clean["search_used"].sum()

# 2. Сколько раз поиск привел к изменению score
changed_score = df_clean["search_changed_score"].sum()

# 3. Сколько раз изменение помогло
helped = df_clean["search_helped"].sum()

# 4. Сколько раз изменение повредило
harmed = df_clean["search_harmed"].sum()

print(f"--- Статистика использования поиска ---")
print(f"Всего использован поиск: {total_searches} раз(а)")
print(f"Поиск изменил score:     {changed_score} раз(а)")
print(f"Изменение помогло:       {helped} раз(а)")
print(f"Изменение навредило:     {harmed} раз(а)")

if total_searches > 0:
    print(f"\n--- Эффективность использования поиска ---")
    print(f"Поиск изменил score:     {changed_score / total_searches:.1%}")
    print(f"Изменение помогло:       {helped / total_searches:.1%}")
    print(f"Изменение навредило:     {harmed / total_searches:.1%}")

--- Статистика использования поиска ---
Всего использован поиск: 99 раз(а)
Поиск изменил score:     29 раз(а)
Изменение помогло:       21 раз(а)
Изменение навредило:     8 раз(а)

--- Эффективность использования поиска ---
Поиск изменил score:     29.3%
Изменение помогло:       21.2%
Изменение навредило:     8.1%


Причины поиска, которые указывали модели, мы никак не используем. 

Но можем посмотреть статистику:

In [45]:
# --- Причины поиска, которые выбирали модели ---
model_responses = [r["result"]["model_responses"] for r in results]
model_responses = [item for sublist in model_responses for item in sublist]

search_reasons = [r["search_reason"] for r in model_responses if r["search_reason"] is not None]
reasons_stats = Counter(search_reasons)

for search_reason, times_requested in sorted(reasons_stats.items()):
    print(f"Причина поиска {search_reason} запрошена {times_requested} раз(а)")

Причина поиска 1 запрошена 170 раз(а)
Причина поиска 2 запрошена 75 раз(а)
Причина поиска 3 запрошена 57 раз(а)


Теперь интересно посмотреть на решения агента, когда среди моделей ансамбля был консенсус.

А это частый случай: 353 раза из 500.

In [46]:
# --- Отбираем те строки, решение ансамбля моделей по которым было консенснусным ---
df_consensus = df_clean[df_clean["consensus"]]
len(df_consensus)

353

Отберем из консенсусных ответов те, в которых агент ошибся.

Это произошло 80 раз - подозрительно много для единогласных решений.

In [47]:
# --- Из строк с консенсусом отбираем те, в которых ансамбль моделей ошибся ---
df_consensus_failed = df_consensus[~df_consensus["matched"]]
len(df_consensus_failed)

80

Из этих 80 раз, 7 раз консенсус наступил только после получения информации через поиск в Интернете - не помог поиск.

In [48]:
# --- Из строк с консенсусом и ошибочным общим решением отбираем те, в которых модели использовали результаты поиска в Интернете ---
df_consensus_failed_with_search = df_consensus_failed[df_consensus_failed["search_used"]]
len(df_consensus_failed_with_search)

7

Посмотрим на примеры этих странных ошибочных консенсусов. 

Первые 5 примеров:

In [49]:
# --- Посмотрим на df_consensus_failed ---
for _, row in df_consensus_failed[:5].iterrows():
    print(f"Запрос пользователя:  {row["query"]}")
    print(f"Наименование объекта: {row["name"]}")
    print(f"Адрес объекта:        {row["address"]}")
    print(f"Поиск использовался:  {row["search_used"]}")
    print(f"Ответ агента:         {row["final_score"]}")
    print(f"Правильный ответ:     {row["true_score"]}\n")

Запрос пользователя:  Эпиляция
Наименование объекта: MaxiLife; Центр красоты и здоровья MaxiLife; Центр красоты и здоровья МаксиЛайф; MaxiLife Health and Beauty Center
Адрес объекта:        Московская область, Одинцово, улица Маршала Жукова, 11А
Поиск использовался:  False
Ответ агента:         0
Правильный ответ:     1

Запрос пользователя:  Купить торт
Наименование объекта: Фабрика Счастье; Lavka schastya; Лавка Счастья; Лавка счастья; Счастье; Lavka Schastya
Адрес объекта:        Санкт-Петербург, Лиговский проспект, 30
Поиск использовался:  False
Ответ агента:         0
Правильный ответ:     1

Запрос пользователя:  где в Астрахани сделать фото радужки глаза
Наименование объекта: Фото радужки глаза; ScreenEyes
Адрес объекта:        Владимир, Большая Московская улица, 16
Поиск использовался:  False
Ответ агента:         0
Правильный ответ:     1

Запрос пользователя:  мильгамма уколы купить в коврове
Наименование объекта: Столички; Stolichki; Аптека Столички; Аптеки Столички; Аптеки 

И 5 примеров таких же единогласных ошибок после применения поиска:

In [50]:
# --- Посмотрим на df_consensus_failed_with_search ---
for _, row in df_consensus_failed_with_search[:5].iterrows():
    print(f"Запрос пользователя:  {row["query"]}")
    print(f"Наименование объекта: {row["name"]}")
    print(f"Адрес объекта:        {row["address"]}")
    print(f"Поиск использовался:  {row["search_used"]}")
    print(f"Ответ агента:         {row["final_score"]}")
    print(f"Правильный ответ:     {row["true_score"]}\n")

Запрос пользователя:  мильгамма уколы купить в коврове
Наименование объекта: Столички; Stolichki; Аптека Столички; Аптеки Столички; Аптеки Столички Ковров Грибоедова; Apteki Stolichki
Адрес объекта:        Владимирская область, Ковров, улица Грибоедова, 28
Поиск использовался:  True
Ответ агента:         1
Правильный ответ:     0

Запрос пользователя:  прием пластика рябиновая
Наименование объекта: Промо-карта; Promo-Karta
Адрес объекта:        Москва, улица Генерала Дорохова, 6, стр. 5
Поиск использовался:  True
Ответ агента:         0
Правильный ответ:     1

Запрос пользователя:  дежурная стоматология севастополь
Наименование объекта: Стандарт; Standart
Адрес объекта:        Севастополь, Большая Морская улица, 33
Поиск использовался:  True
Ответ агента:         1
Правильный ответ:     0

Запрос пользователя:  где купит утку скутер дешевый
Наименование объекта: Мото-скутер; Moto-scuter; Motoscuter; Мотоскутер; Мото-скутер.ру
Адрес объекта:        Санкт-Петербург, проспект Тореза, 68Е

#### ВЫВОДЫ:

1. Превзойти baseline (классификатор на трансформере, обученный на обучающей выборке) удалось, но только на единицы или даже доли процента.

- Точность бейзлайна: 73,20%

- Точность ансамбля из 3-х LLM с two-shot и без поиска: 70,80%, а с поиском: 73,40% - но это один из самых неудачных прогонов, так случайно не повезло.

- Точность ансамбля из 2-х LLM с zero-shot и без поиска: 75,20%, а с поиском: 76,20% - это в блокноте part_3_llm_agent_with_search__m2_with_zero_shot.ipynb. 

2. Думаю, что если ансамбль дешёвых моделей заменить на одну фронтирную LLM, и дать ей совершать более одной итерации поиска за цикл работы агента, то мы получили бы намного более интересные цифры качества.

3. В самих тестовых данных есть вопросы к разметке. В некоторых примерах, когда ансамбль LLM дает единогласный ответ, и который не совпадает с разметкой, то есть сомнения в правильности разметки. Но не берусь утверждать наверняка, не потратил время на изучение таких случаев с пристрастием, не успел.